In [ ]:
from myosuite.utils import gym
import imageio
from IPython.display import Video, display
import numpy as np
import os

In [ ]:
env = gym.make('myoElbowPose1D6MRandom-v0')

env.reset()


In [ ]:
from stable_baselines3 import PPO

model = PPO("MlpPolicy", env, verbose=0)

print("========================================")
print("Starting policy learning")
print("========================================")

model.learn(total_timesteps=1000)

print("========================================")
print("Job Finished.")
print("========================================")

model.save('ElbowPose_policy')


In [ ]:
policy = "ElbowPose_policy.zip"

pi = PPO.load(policy)

# define a discrete sequence of positions to test
AngleSequence = [60, 30, 30, 60, 80, 80, 60, 30, 80, 30, 80, 60]
env.reset()
frames = []
for ep in range(len(AngleSequence)):
    print("Ep {} of {} testing angle {}".format(ep, len(AngleSequence), AngleSequence[ep]))
    env.unwrapped.target_jnt_value = [np.deg2rad(AngleSequence[int(ep)])]
    env.unwrapped.target_type = 'fixed'
    env.unwrapped.weight_range=(0,0)
    env.unwrapped.update_target()
    for _ in range(40):
        frame = env.unwrapped.mj_renderer.render_offscreen(
                        width=400,
                        height=400,
                        camera_id=0)
        frames.append(frame)
        o = env.unwrapped.get_obs()
        a = pi.predict(o)[0]
        next_o, r, done, *_, ifo = env.step(a) # take an action based on the current observation
env.close()

os.makedirs('videos', exist_ok=True)
# make a local copy
imageio.mimwrite(f"videos/arm.mp4", frames, fps=1.0 / env.unwrapped.dt)

display(Video(f"videos/arm.mp4", embed=True))